In [1]:
# ============================================================
# Cell 1: Setup (see Stage 0 Colab header for RFdiffusion install)
# ============================================================
# Run the RFdiffusion setup cell from Section 2.4

# Upload your clean PD-L1 structure
from google.colab import files
# Upload ../data/structures/pdl1_clean.pdb from your local machine
uploaded = files.upload()

# Move to working directory
!mkdir -p input_pdbs
!mv pdl1_clean.pdb input_pdbs/

ModuleNotFoundError: No module named 'google'

In [ ]:
# ============================================================
# Cell 2: Configure and run RFdiffusion binder design
# ============================================================
# CRITICAL: Replace hotspot residues with YOUR refined list from Stage 1
# Format: [ChainID][ResidueNumber] — e.g., A54 means chain A, residue 54
#
# These are EXAMPLES — use your actual hotspot residues from Stage 1:

HOTSPOT_RESIDUES = "A54,A56,A121,A122,A123,A125"  # <-- REPLACE WITH YOUR RESIDUES
BINDER_LENGTH = 70       # residues — start with 70, can try 55-80
NUM_DESIGNS = 100         # number of backbones to generate (start with 100, scale to 500 later)
TARGET_PDB = "input_pdbs/pdl1_clean.pdb"
OUTPUT_DIR = "outputs/rfdiffusion"

!mkdir -p {OUTPUT_DIR}

# Run RFdiffusion in binder design mode
# This generates NUM_DESIGNS backbone structures
# Each takes ~30-60 seconds on a T4 GPU
# 100 designs ≈ 1-2 hours

import subprocess

for i in range(NUM_DESIGNS):
    cmd = f"""python scripts/run_inference.py \
        inference.output_prefix={OUTPUT_DIR}/design_{i} \
        inference.input_pdb={TARGET_PDB} \
        'contigmap.contigs=[{BINDER_LENGTH}-{BINDER_LENGTH}/0 A1-200]' \
        'ppi.hotspot_res=[{HOTSPOT_RESIDUES}]' \
        inference.num_designs=1 \
        denoiser.noise_scale_ca=0 \
        denoiser.noise_scale_frame=0"""

    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"Design {i} failed: {result.stderr[-200:]}")
    elif i % 10 == 0:
        print(f"Completed {i+1}/{NUM_DESIGNS}")

print(f"✓ Generated {NUM_DESIGNS} backbone designs")